In [ ]:
import pandas as pd
import numpy as np
import subprocess
import shutil
import os
import sys
import rally_segment_features as rsf

# === PARAMETERS (tweak as needed) ===
CSV_FILE = 'prediction\picklecam_phq_ball.csv'     # Path to your ball position CSV
FPS = 30                        # Video frame rate (frames per second)
SPEED_THRESHOLD = 5             # px/frame; minimum speed to count as 'in play'
MIN_SEGMENT_LENGTH = 90         # frames; minimum length of a rally to keep (e.g. 3 seconds)
MAX_MERGE_GAP = 40              # frames; merge segments separated by less than this gap
SEGMENT_START_BUFFER = 60       # frames; frames to add to the start of the segment
SEGMENT_END_BUFFER = 10       # frames; frames to add to the end of the segment
INPUT_VIDEO = "data_rally_detection\picklecam_phq.mp4"           # Original input video file
OUTPUT_VIDEO = "prediction\picklecam_phq_highlights.mp4"    # Output file
TMP_DIR = "tmp_rallies"
CRF = 23                            # quality (lower = better)
NET_Y_UPPER = 145
NET_Y_LOWER = 300

# === 1. Read data ===
df = pd.read_csv(CSV_FILE)

total_frames = df.iloc[-1]['Frame']

# === 2. Calculate instantaneous speed (Euclidean distance in (X, Y) pixels) ===
df['dX'] = df['X'].diff()
df['dY'] = df['Y'].diff()
df['Speed'] = np.sqrt(df['dX']**2 + df['dY']**2).fillna(0)

# === 3. Mark frames that are 'active' (ball visible and moving) ===
df['IsActive'] = (df['Visibility'] == 1) & (df['Speed'] > SPEED_THRESHOLD)

df.to_csv(CSV_FILE.replace(".csv", "_speed.csv"))

initial_segments = rsf.get_initial_segments(df, MAX_MERGE_GAP, MIN_SEGMENT_LENGTH)

# Disabled for now
# lob_merge_segments = rsf.merge_lob_segments(initial_segments, df, FPS)

segments_df = rsf.compute_rally_features(df, initial_segments, FPS, NET_Y_UPPER, NET_Y_LOWER,)

segments_df['StartTime_sec'] = segments_df['StartFrame'] / FPS
segments_df['EndTime_sec']   = segments_df['EndFrame'] / FPS

# Filter rules to get highlights
filtered_segments = segments_df[
    (segments_df['Duration_sec'] >= 6) &
    (segments_df['NetCrossings'] >= 2)
].reset_index(drop=True)

# Add buffer to start and end of segments
filtered_segments['StartFrameWithBuffer'] = (filtered_segments["StartFrame"] - SEGMENT_START_BUFFER).clip(lower=0)
filtered_segments['EndFrameWithBuffer'] = (filtered_segments["EndFrame"] + SEGMENT_START_BUFFER).clip(lower=0)

filtered_segments['StartTimeWithBuffer_sec'] = filtered_segments['StartFrameWithBuffer'] / FPS
filtered_segments['EndTimeWithBuffer_sec']   = filtered_segments['EndFrameWithBuffer'] / FPS

# Make sure no segments are overlaping
filtered_segments = (
    filtered_segments
    .sort_values('StartFrameWithBuffer')
    .reset_index(drop=True)
)

next_start = filtered_segments['StartFrameWithBuffer'].shift(-1)           
new_end    = np.where(
    filtered_segments['EndFrameWithBuffer'] >= next_start,                 
    np.maximum(next_start - 1, filtered_segments['StartFrameWithBuffer']), 
    filtered_segments['EndFrameWithBuffer']                                
)

filtered_segments['EndFrameWithBuffer'] = new_end.astype(int)
filtered_segments['EndTimeWithBuffer_sec'] = filtered_segments['EndFrameWithBuffer'] / FPS

filtered_segments.to_csv('detected_rallies.csv', index=False)

In [ ]:


# # === 7. Print segments as timecodes from segments_df ===
# for i, row in segments_df.iterrows():
#     start = row['StartFrame']
#     end   = row['EndFrame']
#     start_time = start / FPS
#     end_time = end / FPS
#     print(f"Rally {i:2d}: Frames {start}–{end} | Time {start_time:.2f}s–{end_time:.2f}s | Duration: {end_time - start_time:.2f}s")


# === 8. (Optional) Save segments to DataFrame for further processing ===
# segments_df = pd.DataFrame({
#     'StartFrame': [start for start, end in final_segments],
#     'EndFrame': [end for start, end in final_segments],
#     'StartTime_sec': [start / FPS for start, end in final_segments],
#     'EndTime_sec': [end / FPS for start, end in final_segments],
#     'Duration_sec': [(end - start + 1) / FPS for start, end in final_segments]
# })




In [ ]:

if os.path.isdir(TMP_DIR):
    for filename in os.listdir(TMP_DIR):
        file_path = os.path.join(TMP_DIR, filename)
        try:
            if os.path.isfile(file_path) or os.path.islink(file_path):
                os.unlink(file_path)  # remove file or symlink
            elif os.path.isdir(file_path):
                shutil.rmtree(file_path)  # remove subdirectory
        except Exception as e:
            print(f'Failed to delete {file_path}. Reason: {e}')
else:
    os.makedirs(TMP_DIR)

segment_files = []
for i, row in filtered_segments.iterrows():
    start     = row['StartTimeWithBuffer_sec']
    duration  = row['EndTimeWithBuffer_sec'] - row['StartTimeWithBuffer_sec']
    seg_path  = os.path.join(TMP_DIR, f"rally_{i:03d}.mp4")

    # Re-encode so we never depend on key-frame alignment
    cmd = [
        "ffmpeg", "-y",
        "-ss", f"{start:.3f}",        # fast → put before -i
        "-i",  INPUT_VIDEO,
        "-t",  f"{duration:.3f}",
        "-r",  str(FPS),              # force constant frame rate
        "-vf", f"fps={FPS}",
        "-c:v", "libx264", "-preset", "veryfast", "-crf", str(CRF),
        "-c:a", "aac", "-b:a", "128k",
        "-movflags", "+faststart",    # nicer for streaming
        seg_path
    ]

    print("Extracting", seg_path)
    res = subprocess.run(cmd, capture_output=True, text=True)
    if res.returncode != 0:
        print(res.stderr)
        sys.exit(f"FFmpeg failed on segment {i}")
    segment_files.append(seg_path)

# ---------- CONCAT ----------
list_file = os.path.join(TMP_DIR, "segments.txt")
with open(list_file, "w") as f:
    for s in segment_files:
        f.write(f"file '{os.path.abspath(s)}'\n")

concat_cmd = [
    "ffmpeg", "-y",
    "-f", "concat", "-safe", "0",
    "-i", list_file,
    "-c", "copy",                 # streams already CFR & compatible
    OUTPUT_VIDEO
]
print("Concatenating segments …")
subprocess.run(concat_cmd, check=True)

print(f"\n✅  Done! Final highlight video: {OUTPUT_VIDEO}")
